In [1]:
from qm import QuantumMachinesManager
import time
import matplotlib.pyplot as plt
from configuration import *
import numpy as np
import scipy.optimize as opti
from auto_mixer_tools_visa import KeysightXSeries
import pyvisa as visa
from qm import SimulationConfig
from qm.qua import *
from qm import LoopbackInterface
from helper_functions import *


qmm = QuantumMachinesManager()
qm = qmm.open_qm(config)

address = "TCPIP0::192.168.0.102::inst0::INSTR"
calib = KeysightXSeries(address, qm)

with program() as mixer_calibration_pulse:
    with infinite_loop_():
        
        play("const"*amp(0.0), "qubit")
        play("const"*amp(0.0), "rr")
        
job = qm.execute(mixer_calibration_pulse)

2022-06-09 18:46:20,395 - qm - INFO - Performing health check
2022-06-09 18:46:20,402 - qm - INFO - Health check passed
2022-06-09 18:46:20,528 - qm - INFO - Flags: 
2022-06-09 18:46:20,528 - qm - INFO - Sending program to QOP
2022-06-09 18:46:20,540 - qm - INFO - Executing program
2022-06-09 18:46:20,607 - qm - INFO - Flags: 
2022-06-09 18:46:20,608 - qm - INFO - Sending program to QOP
2022-06-09 18:46:20,621 - qm - INFO - Executing program


In [2]:
def set_IQ_imbalance_get_leakage(imbalance_arr, qe, qe_freq,verbose=False):
    
    A_imb, P_imb = imbalance_arr
    qe_LO, qe_IF, mixer = qe_freq[qe]
    
    qm.set_mixer_correction(mixer, int(qe_IF), int(qe_LO), IQ_imbalance(A_imb, P_imb))
    
    time.sleep(1)
    
    leakage = calib.query_marker(1)
    if verbose:
        print("Current leakage is {0} dBm".format(leakage))
    
    if leakage < -95:
         return -300
        
    return leakage

In [3]:
qe_freq = {"qubit": [qubit_LO, qubit_IF, "mixer_qubit"],
            "rr": [rr_LO, rr_IF, "mixer_ro"]
          }
qe = "qubit"

In [4]:
with program() as mixer_calibration_pulse:
    with infinite_loop_():
        
        play("const"*amp(1.0), qe)

job = qm.execute(mixer_calibration_pulse)

qe_LO, qe_IF, mixer = qe_freq[qe]
center_freq = qe_LO - qe_IF #rr_LO
span = 50
calib.set_bandwidth(5)
calib.set_sweep_points(501)

calib.set_center_freq(center_freq)
calib.set_span(span)
calib.active_marker(1)
calib.set_marker_freq(1, center_freq)

2022-06-09 18:46:33,221 - qm - INFO - Flags: 
2022-06-09 18:46:33,222 - qm - INFO - Sending program to QOP
2022-06-09 18:46:33,237 - qm - INFO - Executing program


In [ ]:
from scipy.optimize import minimize

init_vals = [0.0, 0.0]
#lower_bound = -60
bnds = ((-0.3, 0.3), (-0.3, 0.3))

xatol = 1e-4  # 1e-4 change in DC offset or gain/phase
#fatol = 3  # dB change tolerance
maxiter = 50  # 50 iterations should be more then enough, but can be changed.
initial_simplex = np.zeros([3, 2])

initial_simplex[0, :] = [0, -0.2]
initial_simplex[1, :] = [0.2, 0.2]
initial_simplex[2, :] = [-0.2, 0.2]

args = [qe, qe_freq]
res = minimize(set_IQ_imbalance_get_leakage, 
               init_vals, 
               args= (qe, qe_freq), 
               method="nelder-mead", 
               bounds=bnds, 
               options={
                "xatol": xatol,
#                "fatol": fatol,
                "initial_simplex": initial_simplex,
                "maxiter": maxiter,
               }
              )

print("Final reject band leakage is {0} dBm".format(calib.query_marker(1)))

center_freq = qe_LO + qe_IF
calib.set_center_freq(center_freq)
calib.set_marker_freq(1, center_freq)
print("Upconverted sideband power is {0} dBm".format(calib.query_marker(1)))

print(f"Calibrated IQ imbalance tuple is {res.x}")

In [ ]:
print(f"Calibrated IQ imbalance tuple is {res.x}")

In [ ]:
def update_config_correctionmatrix(file,mixer,IQval):
    '''
    Used to update IQ values of mixers in configuration file.
    Input example ('configuration.py',"mixer_qubit",[-0.123,-0.035])
    '''

    def line_num_for_phrase_in_file(phrase, filename):
        with open(filename,'r') as f:
            for (i, line) in enumerate(f):
                if phrase in line:
                    return i
        return -1

    ofile = open(file,'r')
    lines = ofile.readlines()
    relev_linenum = line_num_for_phrase_in_file(f'"{mixer}": [',file)
    oldiqvals = lines[relev_linenum].split('(')[1].split(')')[0]
    lines[relev_linenum]=lines[relev_linenum].replace(f'IQ_imbalance({oldiqvals})',f'IQ_imbalance({IQval[0]},{IQval[1]})')

    ofile = open(file,'w')
    ofile.writelines(lines)
    ofile.close()
    return print(f'Config updated for "{mixer}".')

file='configuration.py'
update_config_correctionmatrix(file,mixer,res.x)